# 11. Overfitting — When the System Memorizes Instead of Learns

**Building a Heart Disease Risk-Screening System — Notebook 11 of 12, Stage 5: Building and Validating the Predictive Core**

Notebook 10 measured a train/test gap and called a small one "reassuring." This
notebook makes that gap misbehave on purpose, using a more flexible model
(a decision tree) on the same disease-prediction task, so the failure mode is
visible before it ever happens by accident in a real deployment.

## The topic

Every model family has a **complexity knob** — polynomial degree, tree depth,
number of neighbors. Turn it up and training performance improves, essentially
always. Test performance improves for a while, then stalls or gets *worse* — the
model has stopped learning the real age/cholesterol/chest-pain-type relationship
to disease and started memorizing quirks of these specific 350 training patients.

## Why it matters for this system

An overfit risk model isn't a subtle technical flaw — it's the difference between
a screening tool that generalizes to next month's patients and one that quietly
mis-scores them, with no way to tell from the training numbers alone that anything
is wrong. Test performance is the only honest signal, which is exactly why
Notebook 10's split exists.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

df = pd.read_csv("../5. MLOps/2. End-to-End ML/data/heart_disease_cleaned_2.csv", index_col=0)
inputs = ["age", "sex", "cp", "trestbps", "chol", "thalach", "exang"]
X, y = df[inputs], df["target"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train: {len(X_train)}   Test: {len(X_test)}")

## The toolkit

| Approach | How it controls complexity |
|---|---|
| **Limit model flexibility directly** | Cap tree depth, polynomial degree, or neighbor count |
| **Regularization** | Let the model stay flexible but penalize it for using that flexibility (Ridge/Lasso-style) |
| **More training data** | Makes memorizing noise harder without touching the model at all |
| **The train/test gap itself** | An early-warning diagnostic, not a fix — tells you *whether* you have a problem |

## How to choose

Watch the train/test gap first (a diagnostic, always worth checking regardless of
model family) — it tells you whether overfitting is happening before you decide
what to do about it. If it's happening, limiting complexity directly is the
simplest fix and the right first move on a registry this size (438 patients isn't
enough to support very flexible models safely). Regularization is the better choice
when you want to keep a flexible model but rein in *how much* it uses that
flexibility rather than hard-capping it. Collecting more data helps but usually
isn't available on demand for a clinical registry — the other two levers are what
you actually control day to day.

## Applied to the registry

### Sweeping tree depth: watch training and test performance diverge

In [ ]:
depths = range(1, 16)
train_scores, test_scores = [], []

for d in depths:
    tree = DecisionTreeClassifier(max_depth=d, random_state=0).fit(X_train, y_train)
    train_scores.append(roc_auc_score(y_train, tree.predict_proba(X_train)[:, 1]))
    test_scores.append(roc_auc_score(y_test, tree.predict_proba(X_test)[:, 1]))

results = pd.DataFrame({"depth": depths, "train_AUC": train_scores, "test_AUC": test_scores})
print(results.round(3).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(depths, train_scores, "o-", label="Training AUC", color="steelblue")
ax.plot(depths, test_scores, "o-", label="Test AUC", color="darkorange")
ax.set_xlabel("tree depth"); ax.set_ylabel("AUC")
ax.set_title("Training AUC keeps climbing; test AUC stalls, then degrades")
ax.legend()
plt.show()

Training AUC climbs toward a perfect score as depth increases — a sufficiently
deep tree can eventually memorize every training patient's exact outcome. Test AUC
tells a different story: it improves for a few levels of depth, then flattens or
drops. That divergence point is where the tree stops learning genuine
age/cholesterol/chest-pain patterns and starts fitting noise specific to the 350
training patients.

### The train/test gap as an explicit overfitting alarm

In [ ]:
results["gap"] = results["train_AUC"] - results["test_AUC"]
print(results.round(3).to_string(index=False))

best_depth = results.loc[results["test_AUC"].idxmax(), "depth"]
print(f"\nBest test AUC at depth {best_depth} -- notice how the gap widens well past that point,")
print("giving an early warning before test performance itself turns down.")

### Seeing it directly: decision boundaries at different depths

Using just two inputs (for a plot we can actually see), compare a shallow tree's
boundary to a deep one's.

In [ ]:
X2 = df[["age", "chol"]].to_numpy()
X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y, test_size=0.2, random_state=42, stratify=y)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, d in zip(axes, [1, 3, None]):
    tree2 = DecisionTreeClassifier(max_depth=d, random_state=0).fit(X2_train, y2_train)
    xx, yy = np.meshgrid(np.linspace(X2[:,0].min(), X2[:,0].max(), 200),
                          np.linspace(X2[:,1].min(), X2[:,1].max(), 200))
    Z = tree2.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1].reshape(xx.shape)
    ax.contourf(xx, yy, Z, levels=20, cmap="RdBu_r", alpha=0.6)
    ax.scatter(X2_test[:,0], X2_test[:,1], c=y2_test, cmap="RdBu_r", edgecolor="k", s=25)
    test_auc_d = roc_auc_score(y2_test, tree2.predict_proba(X2_test)[:, 1])
    depth_label = d if d else "unlimited"
    ax.set_title(f"depth={depth_label}\ntest AUC={test_auc_d:.3f}")
    ax.set_xlabel("age"); ax.set_ylabel("chol" if ax is axes[0] else "")
plt.tight_layout(); plt.show()

The unlimited-depth tree's decision boundary fragments into small, jagged
regions chasing individual training points — visually the same phenomenon as
Notebook 9's wild polynomial curve at high degree, just for a classifier's decision
surface instead of a regression line.

### Regularization-style fix: limiting leaf size instead of depth alone

In [ ]:
tree_unregularized = DecisionTreeClassifier(max_depth=None, random_state=0).fit(X_train, y_train)
tree_regularized = DecisionTreeClassifier(max_depth=None, min_samples_leaf=15, random_state=0).fit(X_train, y_train)

for name, t in [("Unregularized (unlimited depth)", tree_unregularized),
                ("min_samples_leaf=15", tree_regularized)]:
    auc = roc_auc_score(y_test, t.predict_proba(X_test)[:, 1])
    print(f"{name:32} test AUC = {auc:.3f}")

`min_samples_leaf` refuses to create a leaf covering too few patients — a
different lever than depth for the same underlying goal: preventing the tree from
carving out a decision region around a handful of training points.

## Systems view — what this stage hands to the next one

We now have a direct, visual, numeric handle on when this system crosses from
learning to memorizing. Notebook 12 formalizes *why* this happens — decomposing the
error into the two competing forces (bias and variance) that every complexity knob
in this notebook was secretly trading off against each other.

## Try it yourself

1. Repeat the depth sweep using only `age` and `sex` as inputs (dropping the
   others) — does overfitting set in at a similar depth, earlier, or later, with
   fewer inputs available to memorize against?
2. Reduce the training set to 100 patients (`X_train[:100]`, `y_train[:100]`) and
   re-run the depth sweep — does overfitting appear at a *lower* depth with less
   training data?
3. Sweep `min_samples_leaf` from 1 to 50 at `max_depth=None` and plot test AUC
   against it — is there a clear "too little, too much" regularization pattern,
   similar to the depth sweep's shape?